In [3]:
!pip uninstall numpy pandas -y
!pip install --no-cache-dir numpy pandas

Found existing installation: numpy 2.2.0
Uninstalling numpy-2.2.0:
  Successfully uninstalled numpy-2.2.0
Found existing installation: pandas 2.3.0
Uninstalling pandas-2.3.0:
  Successfully uninstalled pandas-2.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 19.5 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.3.1 which is incompatible.


In [4]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv('/Users/garvit/Desktop/NestWise/5 - Feature & Model Selection, productionalization/gurgaon_properties_post_feature_selection.csv')
df.head()

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category,price
0,0.0,84.0,4.0,5.0,4.0,0.0,2114.0,1,0,1,2.0,2.0,2.00
1,0.0,85.0,3.0,3.0,4.0,1.0,1900.0,1,0,0,0.0,2.0,1.69
2,0.0,91.0,3.0,4.0,4.0,1.0,1600.0,1,0,1,2.0,0.0,2.10
3,0.0,79.0,3.0,3.0,3.0,3.0,2047.0,0,0,1,0.0,1.0,2.65
4,0.0,2.0,3.0,2.0,2.0,0.0,1502.0,0,0,0,1.0,2.0,0.75


Apply linear reg but first, ordinal encoder to OHE for all categorical col 

In order to do linear reg:
1- One Hot encoding
2- Scaling
3- price col is right skewed -- > apply log transformation

In [55]:
X = df.drop(columns = ['price'])
y = df['price']

In [56]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR

In [57]:
columns_to_encode = ['sector','balcony','agePossession','furnishing_type','luxury_category','floor_category']

In [58]:
y_transformed = np.log1p(y)

In [64]:
preprocessor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(),['property_type','bedRoom','bathroom','built_up_area','servant room','store room']),
        ('cat',OneHotEncoder(drop='first',handle_unknown='ignore'),columns_to_encode + ['sector'])
    ],
    remainder="passthrough"
)

In [65]:
print("Column causing warning:", (columns_to_encode + ['property_type'])[0])

Column causing warning: sector


In [66]:
print(X['sector'].value_counts())

sector
116.0    163
6.0      107
101.0    107
109.0    100
82.0      93
        ... 
104.0      3
34.0       2
115.0      2
46.0       1
25.0       1
Name: count, Length: 117, dtype: int64


In [67]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', SVR(kernel='rbf'))
])

In [68]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

/Users/garvit/Desktop/NestWise/nestwise-env/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/garvit/Desktop/NestWise/nestwise-env/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [69]:
scores.mean()

np.float64(0.8824432626781121)

In [70]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [71]:
pipeline.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [72]:
y_pred = pipeline.predict(X_test)
y_pred = np.expm1(y_pred)

/Users/garvit/Desktop/NestWise/nestwise-env/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [73]:
from sklearn.metrics import mean_absolute_error
mean_absolute_error(np.expm1(y_test),y_pred)

0.55340431106649